# Early CVD Prediction - ML Lab Supervisor Demo Notebook

This is the clean final notebook for demonstrating the **Early CVD Prediction** project to a supervisor. It is intentionally different from the original exploratory neural-network notebook: the old notebook is preserved as historical work, while this notebook shows the actual reproducible project that was built around it.

The notebook follows the ML Lab report structure:

1. Abstract and project framing
2. Introduction and literature positioning
3. Data audit and dataset discrepancy analysis
4. Methodology
5. Experimentation
6. Results and discussion
7. Ablation / comparative studies
8. Explainability and local prediction
9. Localhost application and project architecture
10. Conclusion and future work

Important caution: this is a public-dataset research prototype. It is **not** a clinical diagnostic device.


## Abstract

Cardiovascular disease prediction is a common applied machine-learning project, but public benchmark studies can easily overstate results when duplicate rows, preprocessing leakage, test-set tuning, and calibration are not handled correctly. This project converts an original neural-network notebook into a reproducible heart disease prediction system using a clean Python package, leakage-safe pipelines, site-aware validation, explainability, and a localhost FastAPI app.

The primary dataset is `heart_disease_uci.csv`, which contains 920 records from four centers. The binary target is defined as heart disease absent for `num = 0` and present for `num > 0`. The secondary dataset `heart.csv` is used only as a supplementary benchmark after duplicate analysis. Candidate models include penalized logistic regression, random forest, gradient boosting, calibrated gradient boosting, and an MLP baseline based on the original notebook. The final selected model is penalized logistic regression because it provides the best balance of calibration, interpretability, and site-aware validation performance.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

# Find the repository root whether this notebook is run from the root or notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
        PROJECT_ROOT = candidate
        break

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from domain.feature_dictionary import FEATURE_COLUMNS, GROUP_COLUMN, TARGET_BINARY, TARGET_MULTICLASS
from domain.risk import risk_category, risk_message
from evaluation.explain import local_explanation
from infrastructure.config import load_config
from infrastructure.data import load_primary_dataset, load_secondary_dataset
from infrastructure.persistence import load_bundle

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

config = load_config("configs/default.yaml")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIG_DIR = REPORTS_DIR / "figures"
TABLE_DIR = REPORTS_DIR / "tables"
DOCS_DIR = PROJECT_ROOT / "docs"
MODEL_BUNDLE = PROJECT_ROOT / config["paths"]["model_bundle"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Project name: {config['project_name']}")
print(f"Configured model bundle: {MODEL_BUNDLE}")


## Proposal Alignment

The project proposal was titled **An Explainable Deep Learning Framework for Early Prediction of Cardiovascular Disease Using Clinical Data**. Its main ideas were retained:

- early CVD / heart disease prediction from clinical variables,
- a deep neural-network baseline,
- comparison with classical machine-learning models,
- explainable AI for global and local interpretation,
- robustness checks for missing values, duplicates, overfitting, and class balance,
- a user-facing decision-support style interface.

The only major change is the dataset. The proposal mentioned the Framingham dataset, but the final project uses the provided `heart_disease_uci.csv` and `heart.csv` files. This is scientifically acceptable because the UCI-style file contains hospital/center identifiers, which makes site-aware internal-external validation possible. The final claims are therefore limited to this public benchmark setting.


## Original Notebook Compatibility

The original notebook is useful because it shows the starting point of the project: feature exploration, one-hot encoding, scaling, and feedforward neural-network experiments. However, it is not the final scientific evidence.

Main reasons:

- it used a local 303-row `heart.csv` file rather than the currently uploaded 1,025-row `heart.csv`,
- the uploaded `heart.csv` has extensive exact duplicates,
- neural-network tuning used the test set as validation data,
- the main evaluation was a single random split,
- preprocessing and reporting were notebook-state dependent.

The final project preserves the neural-network idea as `mlp_notebook_baseline`, then upgrades the work into leakage-safe package code with site-aware validation and a deployable localhost app.


In [ ]:
critical_files = [
    DATA_DIR / "heart_disease_uci.csv",
    DATA_DIR / "heart.csv",
    DOCS_DIR / "current_work_audit.md",
    DOCS_DIR / "data_audit_report.md",
    DOCS_DIR / "leakage_audit_report.md",
    DOCS_DIR / "proposal_alignment_and_notebook_compatibility.md",
    REPORTS_DIR / "manuscript" / "early_cvd_ieee_lab_report.docx",
    REPORTS_DIR / "model_card.md",
    MODEL_BUNDLE,
]

artifact_status = pd.DataFrame(
    {
        "artifact": [str(p.relative_to(PROJECT_ROOT)) for p in critical_files],
        "exists": [p.exists() for p in critical_files],
        "size_kb": [round(p.stat().st_size / 1024, 1) if p.exists() else None for p in critical_files],
    }
)
display(artifact_status)

missing = artifact_status.loc[~artifact_status["exists"], "artifact"].tolist()
if missing:
    raise FileNotFoundError("Missing required project artifacts. Run train_project.bat first: " + ", ".join(missing))


## Data Audit and Dataset Discrepancy Analysis

Primary research dataset: `heart_disease_uci.csv`.

Secondary benchmark dataset: `heart.csv`.

Key schema differences handled by the project:

- UCI-style data uses `thalch`; the project standardizes it to `thalach`.
- UCI-style data uses `num`; the project collapses it into binary `target_binary`.
- `heart.csv` uses `target` and numeric-coded categories.
- UCI-style data contains the `dataset` source field; the project standardizes it as `center` and uses it only for validation grouping, not prediction.


In [ ]:
primary = load_primary_dataset(DATA_DIR / "heart_disease_uci.csv")
secondary_dedup = load_secondary_dataset(DATA_DIR / "heart.csv", deduplicate=True)
secondary_no_dedup = load_secondary_dataset(DATA_DIR / "heart.csv", deduplicate=False)
secondary_raw = pd.read_csv(DATA_DIR / "heart.csv")

summary = pd.DataFrame(
    [
        {
            "dataset": "heart_disease_uci.csv",
            "role": "primary research dataset",
            "rows": len(primary),
            "columns_after_standardization": primary.shape[1],
            "exact_duplicate_rows": int(primary.duplicated().sum()),
            "centers": primary[GROUP_COLUMN].nunique(),
            "target_absent": int((primary[TARGET_BINARY] == 0).sum()),
            "target_present": int((primary[TARGET_BINARY] == 1).sum()),
        },
        {
            "dataset": "heart.csv",
            "role": "secondary supplementary benchmark",
            "rows": len(secondary_no_dedup),
            "columns_after_standardization": secondary_no_dedup.shape[1],
            "exact_duplicate_rows": int(secondary_raw.duplicated().sum()),
            "centers": secondary_no_dedup[GROUP_COLUMN].nunique(),
            "target_absent": int((secondary_no_dedup[TARGET_BINARY] == 0).sum()),
            "target_present": int((secondary_no_dedup[TARGET_BINARY] == 1).sum()),
        },
        {
            "dataset": "heart.csv after exact deduplication",
            "role": "deduplicated supplementary benchmark",
            "rows": len(secondary_dedup),
            "columns_after_standardization": secondary_dedup.shape[1],
            "exact_duplicate_rows": 0,
            "centers": secondary_dedup[GROUP_COLUMN].nunique(),
            "target_absent": int((secondary_dedup[TARGET_BINARY] == 0).sum()),
            "target_present": int((secondary_dedup[TARGET_BINARY] == 1).sum()),
        },
    ]
)

display(summary)


In [ ]:
print("Primary class balance by center")
class_by_center = pd.crosstab(primary[GROUP_COLUMN], primary[TARGET_BINARY])
class_by_center = class_by_center.rename(columns={0: "absent", 1: "present"})
class_by_center["total"] = class_by_center.sum(axis=1)
class_by_center["prevalence_present"] = (class_by_center["present"] / class_by_center["total"]).round(3)
display(class_by_center)

print("\nPrimary missingness")
missingness = (
    primary[FEATURE_COLUMNS]
    .isna()
    .sum()
    .rename("missing_n")
    .to_frame()
    .assign(missing_percent=lambda d: (d["missing_n"] / len(primary) * 100).round(2))
    .query("missing_n > 0")
    .sort_values("missing_n", ascending=False)
)
display(missingness)

print("\nMulticlass target feasibility check")
multiclass_counts = primary[TARGET_MULTICLASS].value_counts().sort_index().rename_axis("num").reset_index(name="records")
display(multiclass_counts)


In [ ]:
for figure in [
    "class_distribution_primary.png",
    "site_distribution_primary.png",
    "missingness_heatmap.png",
]:
    path = FIG_DIR / figure
    display(Markdown(f"**{figure.replace('_', ' ').replace('.png', '').title()}**"))
    display(Image(filename=str(path)))


## Methodology

The final methodology is designed as prediction-model research, not just a coding exercise.

Workflow:

`raw data -> audit -> canonical schema -> fold-contained preprocessing -> site-aware validation -> model comparison -> calibration-aware selection -> model bundle -> FastAPI app`

Important design choices:

- The binary target is `target_binary = 1` when UCI `num > 0`, otherwise `0`.
- The center/hospital source is used as a validation group and is excluded from the predictor set.
- Numeric variables are median-imputed and standardized inside scikit-learn pipelines.
- Categorical variables are most-frequent-imputed and one-hot encoded inside the same pipelines.
- The main validation is leave-one-center-out internal-external validation.
- Inner model selection is performed only on the development centers.
- Model choice considers AUROC, AUPRC, Brier score, calibration slope, MCC, stability, and interpretability.

This is stronger than a single random split because it tests whether the model generalizes across hospital-like data sources.


In [ ]:
# Optional full retraining cell.
# This can take time. The default notebook path reads already generated reproducible artifacts.
RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    from application.train import train_and_evaluate
    result = train_and_evaluate(config)
    display(result["summary"].head())
else:
    print("Full training skipped. Set RUN_FULL_TRAINING = True to regenerate all tables, figures, and the model bundle.")
    print("Windows shortcut: double-click train_project.bat from the project root.")


## Experimentation

The project compares five models:

1. Penalized logistic regression
2. Random forest
3. Gradient boosting
4. Calibrated gradient boosting
5. MLP notebook baseline

The neural network is included because it was the original direction of the project proposal and old notebook. It is treated as a comparator, not assumed to be the best model.


In [ ]:
required_tables = [
    "model_selection_ranking",
    "model_comparison",
    "center_metrics",
    "bootstrap_confidence_intervals",
    "threshold_analysis",
    "duplicate_sensitivity",
    "supplementary_heart_csv_benchmark",
    "global_feature_importance",
]

tables = {}
for name in required_tables:
    path = TABLE_DIR / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing table: {path}. Run train_project.bat first.")
    tables[name] = pd.read_csv(path)

ranked = tables["model_selection_ranking"].sort_values("selection_score")
champion = ranked.iloc[0]["model"]

cols = [
    "model",
    "selection_score",
    "auroc_mean",
    "auprc_mean",
    "brier_mean",
    "balanced_accuracy_mean",
    "mcc_mean",
    "calibration_slope_mean",
]
display(ranked[cols].round(3))
print(f"Selected champion: {champion}")


In [ ]:
display(Image(filename=str(FIG_DIR / "model_comparison.png")))


## Results and Discussion

The selected final model is **penalized logistic regression**. Calibrated gradient boosting had slightly higher mean AUROC, but logistic regression was selected because it had the best overall selection score after considering calibration, Brier score, MCC, stability, and interpretability.

This is an important scientific point: a more complex model is not automatically better for a small, center-shifted clinical benchmark dataset.


In [ ]:
center = tables["center_metrics"].query("model == @champion").copy()
center_view = center[
    [
        "held_out_center",
        "auroc",
        "auprc",
        "brier",
        "balanced_accuracy",
        "sensitivity",
        "specificity",
        "mcc",
        "calibration_slope",
        "n",
        "prevalence",
    ]
]
display(center_view.round(3))

print("Center-aware mean and standard deviation for champion")
display(center_view.drop(columns=["held_out_center"]).agg(["mean", "std"]).round(3))


In [ ]:
print("Bootstrap confidence intervals for pooled held-out-center predictions")
boot = tables["bootstrap_confidence_intervals"]
display(boot.round(3))

print("Threshold analysis")
threshold_view = tables["threshold_analysis"][[
    "threshold",
    "sensitivity",
    "specificity",
    "precision_ppv",
    "npv",
    "balanced_accuracy",
    "mcc",
]]
display(threshold_view.round(3))


In [ ]:
for figure in [
    "roc_curves.png",
    "precision_recall_curves.png",
    "calibration_plots.png",
    "confusion_matrix_champion.png",
    "decision_curve.png",
]:
    display(Markdown(f"**{figure.replace('_', ' ').replace('.png', '').title()}**"))
    display(Image(filename=str(FIG_DIR / figure)))


## Ablation / Comparative Studies

For this ML Lab project, the ablation section is best presented as **comparative and sensitivity studies**.

What is included:

- Model comparison across logistic regression, random forest, gradient boosting, calibrated boosting, and MLP.
- Duplicate sensitivity on `heart.csv`, comparing duplicate-expanded and deduplicated versions.
- Threshold sensitivity from 0.10 to 0.90.
- Center-wise sensitivity by holding out each hospital/source.

This is more honest than forcing a feature-removal ablation on a small center-shifted dataset.


In [ ]:
print("Supplementary duplicate sensitivity on heart.csv")
supp = tables["supplementary_heart_csv_benchmark"][[
    "dataset_version",
    "auroc_mean",
    "auprc_mean",
    "balanced_accuracy_mean",
    "brier_mean",
    "mcc_mean",
    "note",
]]
display(supp.round(3))

print("Interpretation: the duplicate-expanded file gives higher apparent performance, so it is not used for primary claims.")


## Explainability / XAI

The project includes explainability at two levels:

- Global explanation: which variables are most important overall.
- Local explanation: which variables moved the probability for an individual input profile.

For the final logistic regression model, local explanations are based on transformed feature contributions from the trained pipeline. These explanations are useful for transparency, but they are not causal medical explanations.


In [ ]:
importance = tables["global_feature_importance"].sort_values("importance", ascending=False)
display(importance.head(12).round(3))

display(Markdown("**Global Feature Importance**"))
display(Image(filename=str(FIG_DIR / "global_feature_importance.png")))

display(Markdown("**SHAP-Style Summary**"))
display(Image(filename=str(FIG_DIR / "shap_summary.png")))


In [ ]:
bundle = load_bundle(MODEL_BUNDLE)
metadata = bundle["metadata"]
print("Loaded model from bundle:", metadata["model_name"])
print("Validation design:", metadata["validation_design"])
print("Target definition:", metadata["target_definition"])

sample_patient = pd.DataFrame([
    {
        "age": 58,
        "sex": "Male",
        "cp": "asymptomatic",
        "trestbps": 140,
        "chol": 250,
        "fbs": "false",
        "restecg": "normal",
        "thalach": 135,
        "exang": "true",
        "oldpeak": 2.1,
        "slope": "flat",
        "ca": 1,
        "thal": "reversable defect",
    }
])

probability = float(bundle["model"].predict_proba(sample_patient)[0, 1])
explanation = local_explanation(bundle, sample_patient, top_n=6)

sample_display = sample_patient.T.rename(columns={0: "value"})
display(Markdown("**Example Patient Input**"))
display(sample_display)

display(Markdown(f"**Predicted probability:** {probability:.1%}"))
display(Markdown(f"**Risk category:** {risk_category(probability)}"))
display(Markdown(risk_message(probability)))

display(Markdown("**Why the model predicted this risk**"))
display(pd.DataFrame(explanation))


## Localhost Application

The project includes a FastAPI localhost app with one public prediction workflow. The frontend contains a clinical-style input form, predicted probability, calibrated risk category, explanation panel, model metadata, API documentation link, and a research-prototype disclaimer.

Run on Windows:

```powershell
run_app.bat
```

Then open:

```text
http://127.0.0.1:8000
```

API documentation is available at:

```text
http://127.0.0.1:8000/docs
```

The app uses the same serialized model bundle loaded above, so the notebook demo and web app are connected to the same trained artifact.


In [ ]:
project_inventory = []
for relative_path, description in [
    ("src/domain", "Feature definitions, target names, risk-category rules"),
    ("src/application", "Training, evaluation, and reporting use cases"),
    ("src/infrastructure", "Config, CSV loading, persistence, tracking"),
    ("src/models", "Candidate model factory"),
    ("src/evaluation", "Metrics, calibration, decision curves, explanations"),
    ("src/api", "FastAPI routes and schemas"),
    ("src/webapp", "Frontend templates and static files"),
    ("docs", "Audit and architecture documents"),
    ("reports/figures", "Generated publication-style figures"),
    ("reports/tables", "Generated experiment tables"),
    ("reports/manuscript", "IEEE-style report and manuscript drafts"),
    ("tests", "Unit, integration, and API tests"),
    ("docker", "Docker deployment files"),
    (".github/workflows", "GitHub Actions CI"),
]:
    path = PROJECT_ROOT / relative_path
    project_inventory.append(
        {
            "path": relative_path,
            "exists": path.exists(),
            "items": len(list(path.rglob("*"))) if path.exists() else 0,
            "purpose": description,
        }
    )

display(pd.DataFrame(project_inventory))


## What to Tell the Supervisor

A concise explanation of how the project works:

> I started with a neural-network notebook for heart disease prediction, then audited it and found that it was not strong enough as final scientific evidence because it used a small local dataset, a random split, and test-set validation during tuning. I rebuilt the work as a reproducible ML project. The primary dataset is the UCI-style heart disease file because it includes hospital/source information, allowing leave-one-center-out validation. The pipeline standardizes the schema, defines a binary target, keeps imputation and encoding inside cross-validation folds, compares classical ML models and an MLP baseline, and selects the final model using calibration, discrimination, stability, and interpretability. Logistic regression was selected because it was more stable and interpretable than the neural-network baseline for this small dataset. The final model is saved as a bundle and served through a FastAPI localhost app with probability, risk category, and explanation output.


## Conclusion and Future Work

This project meets the ML Lab requirements by combining a runnable notebook, a clean codebase, generated reports, comparative studies, explainability, and a localhost app. It is stronger than the original notebook because it avoids duplicate leakage, avoids test-set tuning, uses center-aware validation, reports calibration and threshold behavior, and keeps model claims honest.

Future work should include larger modern cohorts, prospective validation, fairness analysis, more robust missingness modeling, and clinician-centered usability testing. Until that is done, the system should be presented only as an educational and research prototype.


## References

1. Collins GS et al. TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. BMJ, 2024.
2. Collins GS et al. PROBAST+AI: an updated quality, risk of bias, and applicability assessment tool for prediction models using regression or artificial intelligence methods. BMJ, 2025.
3. Detrano R et al. International application of a new probability algorithm for the diagnosis of coronary artery disease. American Journal of Cardiology, 1989.
4. Pedregosa F et al. Scikit-learn: Machine Learning in Python. Journal of Machine Learning Research, 2011.
5. Lundberg SM and Lee SI. A Unified Approach to Interpreting Model Predictions. NeurIPS, 2017.
